# Solubility Prediction — Example Notebook

This notebook demonstrates the **SolubilityPredict** package by running predictions on the Lovric dataset (non-overlapping subset with AqSolDB) and evaluating against measured values.

**Dataset**: `data/Lovric2020_logS0_nonoverlap.csv` — 275 molecules not present in the training set  
**Model**: Pre-trained Random Forest (`data/model.pkl`), trained on AqSolDB 80/20 split

## 1. Setup

In [ ]:
import sys
import os

# Add the SolubilityPredict package root to path
pkg_root = os.path.abspath(".")
if pkg_root not in sys.path:
    sys.path.insert(0, pkg_root)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

from src.predict import predict, load_model

sns.set_theme(style="whitegrid", font_scale=1.1)

# Absolute path to the pre-trained model
MODEL_PATH = os.path.join(pkg_root, "data", "model.pkl")
print("Setup complete.")

## 2. Load the Lovric dataset

In [ ]:
df = pd.read_csv(os.path.join(pkg_root, "data", "Lovric2020_logS0_nonoverlap.csv"))
df = df.rename(columns={"isomeric_smiles": "smiles", "logS0": "Solubility"})
print(f"Total molecules: {len(df)}")
df.head()

In [ ]:
# Check for missing SMILES or solubility values
print(f"Missing SMILES:     {df['smiles'].isna().sum()}")
print(f"Missing Solubility: {df['Solubility'].isna().sum()}")

df = df.dropna(subset=["smiles", "Solubility"]).reset_index(drop=True)
print(f"Molecules after dropping nulls: {len(df)}")

## 3. Run predictions

In [ ]:
model = load_model(MODEL_PATH)

smiles_list = df["smiles"].tolist()
print(f"Predicting solubility for {len(smiles_list)} molecules...")

results = predict(smiles_list, model=model)
print("Done.")

In [ ]:
# Build results dataframe
results_df = pd.DataFrame(results)

# Separate valid predictions from errors
errors_df = results_df[results_df["error"].notna()]
valid_df  = results_df[results_df["error"].isna()].copy()

print(f"Successful predictions: {len(valid_df)}")
print(f"Failed (invalid SMILES): {len(errors_df)}")

if len(errors_df) > 0:
    print("\nSample failed SMILES:")
    print(errors_df[["smiles", "error"]].head())

In [ ]:
# Merge predictions with measured values
valid_df = valid_df.merge(
    df[["smiles", "Solubility"]],
    on="smiles",
    how="left"
).rename(columns={"Solubility": "logS_measured", "logS": "logS_predicted"})

valid_df = valid_df.dropna(subset=["logS_measured"])
valid_df[["smiles", "logS_measured", "logS_predicted", "category"]].head(10)

## 4. Evaluate model performance

In [ ]:
y_true = valid_df["logS_measured"].values
y_pred = valid_df["logS_predicted"].values

rmse = mean_squared_error(y_true, y_pred) ** 0.5
mae  = mean_absolute_error(y_true, y_pred)
r2   = r2_score(y_true, y_pred)

print(f"RMSE : {rmse:.3f} logS units")
print(f"MAE  : {mae:.3f} logS units")
print(f"R²   : {r2:.3f}")

## 5. Parity plot — Measured vs. Predicted

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))

ax.scatter(y_true, y_pred, alpha=0.3, s=8, edgecolors="none", color="steelblue")

lim = [min(y_true.min(), y_pred.min()) - 0.5, max(y_true.max(), y_pred.max()) + 0.5]
ax.plot(lim, lim, "r--", linewidth=1.2, label="Ideal")
ax.set_xlim(lim)
ax.set_ylim(lim)

ax.set_xlabel("Measured logS (Lovric)")
ax.set_ylabel("Predicted logS")
ax.set_title("Solubility Prediction on Lovric Dataset")
ax.text(0.05, 0.93,
        f"RMSE = {rmse:.3f}\nMAE  = {mae:.3f}\nR²   = {r2:.3f}",
        transform=ax.transAxes, fontsize=11,
        verticalalignment="top",
        bbox=dict(boxstyle="round", facecolor="white", alpha=0.7))
ax.legend()
plt.tight_layout()
plt.show()

## 6. Error distribution

In [ ]:
residuals = y_pred - y_true

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Residual histogram
axes[0].hist(residuals, bins=60, color="steelblue", edgecolor="white", linewidth=0.4)
axes[0].axvline(0, color="red", linestyle="--")
axes[0].set_xlabel("Residual (Predicted − Measured)")
axes[0].set_ylabel("Count")
axes[0].set_title("Residual Distribution")

# Residuals vs measured
axes[1].scatter(y_true, residuals, alpha=0.3, s=8, edgecolors="none", color="steelblue")
axes[1].axhline(0, color="red", linestyle="--")
axes[1].set_xlabel("Measured logS")
axes[1].set_ylabel("Residual")
axes[1].set_title("Residuals vs. Measured logS")

plt.tight_layout()
plt.show()

## 7. Solubility category breakdown

In [ ]:
category_counts = valid_df["category"].value_counts()

fig, ax = plt.subplots(figsize=(7, 4))
order = ["highly soluble", "soluble", "moderately soluble", "poorly soluble"]
counts = [category_counts.get(c, 0) for c in order]
colors = ["#2ecc71", "#3498db", "#f39c12", "#e74c3c"]

bars = ax.barh(order, counts, color=colors)
ax.bar_label(bars, padding=4)
ax.set_xlabel("Number of molecules")
ax.set_title("Predicted Solubility Categories")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

print(category_counts.reindex(order))

## 8. Inspect worst predictions

In [ ]:
valid_df["abs_error"] = (valid_df["logS_predicted"] - valid_df["logS_measured"]).abs()

worst = valid_df.nlargest(10, "abs_error")[["smiles", "logS_measured", "logS_predicted", "abs_error"]]
worst.style.format({"logS_measured": "{:.3f}", "logS_predicted": "{:.3f}", "abs_error": "{:.3f}"})

## 9. Predict a custom molecule

In [ ]:
# Replace with any SMILES of interest
custom_smiles = "CC(=O)Nc1ccc(O)cc1"  # Paracetamol (acetaminophen)

result = predict(custom_smiles, model=model)
print(f"SMILES     : {result['smiles']}")
print(f"logS       : {result['logS']}")
print(f"Solubility : {result['solubility_mol_L']} mol/L")
print(f"Category   : {result['category']}")